In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set(style="whitegrid", font_scale=1.2, rc={"figure.figsize": (8, 5)})
np.random.seed(42)


In [ ]:
# Импорт необходимых библиотек для визуализации деревьев
from sklearn.tree import export_graphviz
import graphviz
from IPython.display import display

# Функция для визуализации дерева
def visualize_tree(clf, feature_names=None, class_names=None, title="Decision Tree"):
    """
    Визуализирует дерево решений с помощью graphviz
    """
    dot_data = export_graphviz(
        clf,
        out_file=None,
        feature_names=feature_names if feature_names else [f"x{i+1}" for i in range(clf.n_features_in_)],
        class_names=class_names if class_names else ["Class 0", "Class 1"],
        filled=True,
        rounded=True,
        special_characters=True,
        impurity=True,
        proportion=True
    )
    graph = graphviz.Source(dot_data)
    display(graph)
    return graph

In [ ]:
def generate_simple_binary_data(n_per_class=100):
    """С ШУМОМ: 15% меток перепутаны"""
    np.random.seed(42)

    # Чистые данные
    x0 = np.random.randn(n_per_class, 2) + np.array([0, 0])
    y0 = np.zeros(n_per_class, dtype=int)
    x1 = np.random.randn(n_per_class, 2) + np.array([3, 3])
    y1 = np.ones(n_per_class, dtype=int)

    X = np.vstack([x0, x1])
    y = np.concatenate([y0, y1])

    # ДОБАВЛЯЕМ ШУМ: меняем 15% меток местами
    noise_indices = np.random.choice(len(y), size=int(0.15*len(y)), replace=False)
    y[noise_indices] = 1 - y[noise_indices]  # 0↔1

    return X, y

X, y = generate_simple_binary_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], label="class 0", alpha=0.7)
plt.scatter(X[y==1, 0], X[y==1, 1], label="class 1", alpha=0.7)
plt.legend()
plt.title("Сгенерированные данные (2 класса + 15% ШУМ)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


#**Критерий информативности**


В решающем дереве каждый узел соответствует некоторому подмножеству объектов обучающей выборки.

Пусть у нас задача классификации на K классов. В узле R:

*   лежит N объектов: (xi,yi)
*   из них n1 объектов класса 1, n2 объектов класса 2, ..., nK объектов класса K
*   N=n1+⋯+nK

Определим эмпирические вероятности классов в узле:
$$
p_k = \frac{n_k}{N}, \quad k = 1,\dots,K
$$

Это просто доли классов в данном узле.


Если узел «чистый» — в нём все объекты одного класса — то:

*   для какого-то k: pk=1
*   для остальных k: pk=0


Мы хотим, чтобы дерево разделяло пространство признаков так, чтобы:

*   в листьях объекты в основном были одного класса
*   классификация была простой: «если попали в этот лист, скорее всего, это класс k»

Для этого нужен критерий информативности (gain) - метрика, которая показывает, насколько эффективно конкретный признак разделяет данные на классы.

**Для его подсчета вводят функцию неопределённости (или критерий неоднородности, impurity) узла H(R)**. Её свойства:​​

*   H(R)=0, если узел полностью чистый (все объекты одного класса)
*   H(R) максимальна, когда все классы перемешаны равномерно
*   чем больше H(R), тем «хуже» узел

Тогда качество разбиения родительского узла $R$ на два дочерних $R_\text{left}, R_\text{right}$ можно измерять как **уменьшение неопределённости и, следовательно, прирост информативности Gain:**
$$
\text{Gain} = H(R) - \frac{|R_\text{left}|}{|R|} H(R_\text{left}) - \frac{|R_\text{right}|}{|R|} H(R_\text{right})
$$

Мы ищем разбиение (признак + порог), при котором этот Gain максимален.


"Какое разбиение даст БОЛЬШЕ ВСЕГО пользы?"

Дерево пробует варианты:

*   x1 ≤ 1.2 ? Gain = 0.25
*   x1 ≤ 1.5 ? Gain = **0.42 очка** ← ЛУЧШЕ!
*   x2 ≤ 0.8 ? Gain = 0.18 очков

БЕРЁТ тот, у кого **больше очков**!

#**Основные критерии информативности H(R)**

#Gini impurity
**Формула Gini:**
$$
H_\text{Gini}(R) = 1 - \sum_{k=1}^K p_k^2
$$

**Интерпретация:**  
вероятность того, что если мы случайно выберем **два объекта** из узла, они окажутся разных классов.

Чем больше перемешивание → тем больше эта вероятность

Свойства:

*   Узел чистый: один pk=1, остальные 0 → сумма квадратов = 1 → Gini = 0.
*   Узел максимально перемешан (все классы в равных долях) → Gini максимален.​

In [ ]:
#пример подсчета

import pandas as pd
import numpy as np

def calculate_gini(labels):
    """Расчет коэффициента неопределенности Джини."""
    if len(labels) == 0:
        return 0
    probs = np.bincount(labels) / len(labels)
    return 1 - np.sum(probs**2)

def gini_gain(parent, left_child, right_child):
    """Расчет прироста (уменьшения неопределенности) после разделения."""
    # Вес каждой дочерней ветви
    w_left = len(left_child) / len(parent)
    w_right = len(right_child) / len(parent)

    # Прирост = Gini(до) - (Вес * Gini(лево) + Вес * Gini(право))
    gain = calculate_gini(parent) - (w_left * calculate_gini(left_child) +
                                     w_right * calculate_gini(right_child))
    return gain

# --- Пример данных ---
# Представим, что мы классифицируем: купит ли клиент товар (1 - да, 0 - нет)
target = [1, 1, 0, 0, 1, 1, 0, 0] # Исходный узел (50/50)

# Гипотетический вариант разделения №1
left_1 = [1, 1, 1]
right_1 = [0, 0, 0, 0, 1]

# Разделение №2 (более эффективное): оба узла стали чище
left_2 = [1, 1, 1, 1]
right_2 = [0, 0, 0, 0]

# Расчеты
gain_1 = gini_gain(target, left_1, right_1)
gain_2 = gini_gain(target, left_2, right_2)

print(f"Gini до разделения: {calculate_gini(target):.3f}")
print("-" * 30)
print(f"Вариант №1: Gain = {gain_1:.3f}")
print(f"Вариант №2: Gain = {gain_2:.3f}")

In [ ]:
# Визуализация дерева с Gini критерием
print("="*60)
print("Визуализация дерева с Gini критерием")
print("="*60)

# Обучаем дерево с Gini
clf_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf_gini.fit(X_train, y_train)

# Визуализируем
gini_tree = visualize_tree(
    clf_gini,
    feature_names=['x1', 'x2'],
    class_names=['Class 0', 'Class 1'],
    title="Decision Tree (Gini Impurity)"
)

#Энтропийный критерий

**Энтропия (Shannon):**
$$
H_\text{Ent}(R) = -\sum_{k=1}^K p_k \log_2 p_k
$$

**Интерпретация:** «меры неопределённости» распределения классов.

**Свойства:**
- Узел чистый → один $p_k=1$, остальные 0 → энтропия = 0
- Узел с равномерным распределением классов → энтропия максимальна

**Особенность:** энтропия растёт чуть быстрее, чем Gini, когда распределение становится более равномерным, поэтому она сильнее штрафует «тонкие хвосты» (малые, но ненулевые вероятности неправильных классов).


In [ ]:
def calculate_entropy(labels):
    """Расчет энтропии Шеннона."""
    if len(labels) == 0:
        return 0

    # Считаем доли классов
    probs = np.bincount(labels) / len(labels)

    # Фильтруем нулевые вероятности, так как log2(0) не определен
    probs = probs[probs > 0]

    # Формула энтропии: H = - sum(p * log2(p))
    return -np.sum(probs * np.log2(probs))

def information_gain(parent, left_child, right_child):
    """Расчет прироста информации (Information Gain)."""
    # Вес каждой дочерней ветви
    w_left = len(left_child) / len(parent)
    w_right = len(right_child) / len(parent)

    # IG = Энтропия(до) - (Взвешенная энтропия после)
    gain = calculate_entropy(parent) - (w_left * calculate_entropy(left_child) +
                                        w_right * calculate_entropy(right_child))
    return gain


# --- Пример данных ---
# Представим, что мы классифицируем: купит ли клиент товар (1 - да, 0 - нет)
target = [1, 1, 0, 0, 1, 1, 0, 0] # Исходный узел (50/50)

# Гипотетический вариант разделения №1
left_1 = [1, 1, 1]
right_1 = [0, 0, 0, 0, 1]

# Разделение №2 (более эффективное): оба узла стали чище
left_2 = [1, 1, 1, 1]
right_2 = [0, 0, 0, 0]

ent_parent = calculate_entropy(target)
ig_1 = information_gain(target, left_1, right_1)
ig_2 = information_gain(target, left_2, right_2)

print(f"Энтропия до разделения: {ent_parent:.3f}") # Для 50/50 всегда будет 1.0
print("-" * 35)
print(f"Вариант №1: Gain = {ig_1:.3f}")
print(f"Вариант №2: Gain = {ig_2:.3f}")
print("-" * 35)

if ig_2 > ig_1:
    print("Результат: Выбран вариант №2 (максимальный G)")

In [ ]:
# Визуализация дерева с энтропийным критерием
print("="*60)
print("Визуализация дерева с энтропийным критерием")
print("="*60)

# Обучаем дерево с энтропией
clf_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf_entropy.fit(X_train, y_train)

# Визуализируем
entropy_tree = visualize_tree(
    clf_entropy,
    feature_names=['x1', 'x2'],
    class_names=['Class 0', 'Class 1'],
    title="Decision Tree (Entropy)"
)

# Сравним split'ы
print(f"\nGini дерево - особенности разделения:")
print(f"  Первый сплит: признак {clf_gini.tree_.feature[0]}, порог {clf_gini.tree_.threshold[0]:.3f}")
print(f"Энтропийное дерево - особенности разделения:")
print(f"  Первый сплит: признак {clf_entropy.tree_.feature[0]}, порог {clf_entropy.tree_.threshold[0]:.3f}")

#Связь с методом максимального правдоподобия (MLE)

**Узел = статистическая выборка**

В узле лежат N объектов разных классов.

**MLE говорит:** вероятность класса k = его доля в узле: $p_k = n_k/N$

**Правдоподобие данных =** $\sum p_k \cdot \log(p_k)$ (чем больше, тем лучше)

**Энтропия =** $-\sum p_k \cdot \log(p_k)$ (чем меньше, тем лучше)

**Итог:** дерево **минимизирует энтропию** = **максимизирует правдоподобие**

**Gini** делает то же самое, но проще:  
$H_\text{Gini}(R) = 1 - \sum p_k^2$
(квадраты вероятностей вместо логарифмов — быстрее считается, похожий эффект)


In [ ]:
import numpy as np

def calculate_log_likelihood(labels):
    n = len(labels)
    if n == 0:
        return 0

    # Преобразуем список в массив numpy для математических операций
    labels_np = np.array(labels)

    p = np.mean(labels_np)

    if p == 0 or p == 1:
        return 0

    ll = np.sum(labels_np * np.log(p) + (1 - labels_np) * np.log(1 - p))
    return ll

def mle_gain(parent, left_child, right_child):
    """
    Прирост правдоподобия (Likelihood Gain).
    Показывает, насколько 'правдоподобнее' стало описание данных после разделения.
    """
    # LL до разделения
    ll_parent = calculate_log_likelihood(parent)

    # LL после разделения (сумма правдоподобий двух новых веток)
    ll_children = calculate_log_likelihood(left_child) + calculate_log_likelihood(right_child)

    # Gain здесь — это разница (насколько мы стали ближе к 0)
    return ll_children - ll_parent

# --- Пример данных (ваши данные) ---
target = [1, 1, 0, 0, 1, 1, 0, 0] # Исходный узел

# Вариант №1: Неидеальное разделение
left_1 = [1, 1, 1]
right_1 = [0, 0, 0, 0, 1]

# Вариант №2: Идеальное разделение
left_2 = [1, 1, 1, 1]
right_2 = [0, 0, 0, 0]

# Расчеты
ll_parent = calculate_log_likelihood(target)
gain_1 = mle_gain(target, left_1, right_1)
gain_2 = mle_gain(target, left_2, right_2)

print(f"Log-Likelihood до разделения: {ll_parent:.3f}")
print("-" * 40)
print(f"Вариант №1 (Смешанный): MLE Gain = {gain_1:.3f}")
print(f"Вариант №2 (Идеальный):  MLE Gain = {gain_2:.3f}")
print("-" * 40)

if gain_2 > gain_1:
    print("Результат: Выбран вариант №2 (максимальный прирост правдоподобия)")


#Критерий останова (pre-pruning)

Теперь, когда мы понимаем, что в узле есть неопределённость $H(R)$ и как её считать, поговорим о том, **когда останавливаться** и не делить узел дальше.

Почему нельзя делить до бесконечности

Если на каждом шаге мы будем продолжать делить узлы, пока хоть немного уменьшается H(R), дерево:
- станет очень глубоким
- начнёт подгоняться под шум
- может иметь листья с 1–2 объектами

**Это приводит к переобучению:** на обучении почти нет ошибок, на тесте ошибки большие.

#Основные критерии останова

Вводятся простые условия, при которых разбиение узла **запрещается**:

| Параметр | Описание |
|----------|----------|
| `max_depth` | ограничение максимальной глубины дерева |
| `min_samples_split` | минимальное число объектов в узле, при котором ещё можно делить |
| `min_samples_leaf` | минимальное число объектов в листе |
| `min_impurity_decrease` | разбиение выполняется, только если уменьшение H(R) достаточно большое |

**Идея:** мы не даём дереву стать «слишком детализированным».


In [ ]:
# демонстрация критериев останова
from sklearn.tree import DecisionTreeClassifier
def train_and_show_tree(max_depth=None, min_samples_leaf=1):
    """Обучает дерево и показывает метрики + структуру"""
    clf = DecisionTreeClassifier(
        criterion="gini",
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    clf.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc = accuracy_score(y_test, clf.predict(X_test))

    print(f"\n{'='*50}")
    print(f"max_depth={max_depth}, min_samples_leaf={min_samples_leaf}")
    print(f"ФАКТИЧЕСКАЯ глубина: {clf.get_depth()}")
    print(f"Количество листьев: {clf.get_n_leaves()}")
    print(f"Trein accuracy: {train_acc:.3f}")
    print(f"Test accuracy:  {test_acc:.3f}")
    print(f"{'='*50}")

# Тестируем разные настройки
print("Критерии останова")
settings = [
    (None, 1),    # Без ограничений
    (1, 1),       # Только 1 уровень
    (3, 1),       # 3 уровня
    (None, 5),    # Минимум 5 в листе
    (None, 10),   # Минимум 10 в листе
]

for max_d, min_leaf in settings:
    train_and_show_tree(max_d, min_leaf)

#Переобучение и стрижка деревьев

In [ ]:
#Построим очень глубокое дерево (почти без ограничений) и посмотрим на разницу между train и test.

from sklearn.metrics import accuracy_score

def train_tree(max_depth=None, min_samples_leaf=1):
    clf = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    clf.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc = accuracy_score(y_test, clf.predict(X_test))

    return clf, train_acc, test_acc

clf_big, train_acc_big, test_acc_big = train_tree(max_depth=None, min_samples_leaf=1)
clf_small, train_acc_small, test_acc_small = train_tree(max_depth=3, min_samples_leaf=5)

print("Большое дерево:")
print("  depth =", clf_big.get_depth(), "leaves =", clf_big.get_n_leaves())
print("  train_acc =", train_acc_big, "test_acc =", test_acc_big)

print("\nОграниченное дерево:")
print("  depth =", clf_small.get_depth(), "leaves =", clf_small.get_n_leaves())
print("  train_acc =", train_acc_small, "test_acc =", test_acc_small)


In [ ]:
# Сравнение переобученного и хорошо обобщающего дерева
print("="*60)
print("Сравнение переобученного и хорошо обобщающего дерева")
print("="*60)

# Переобученное дерево
clf_overfit = DecisionTreeClassifier(max_depth=10, min_samples_leaf=1, random_state=42)
clf_overfit.fit(X_train, y_train)

# Хорошо обобщающее дерево
clf_good = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
clf_good.fit(X_train, y_train)

print("\n⚠️ ПЕРЕОБУЧЕННОЕ дерево (max_depth=10, min_samples_leaf=1):")
print(f"   Train accuracy: {accuracy_score(y_train, clf_overfit.predict(X_train)):.3f}")
print(f"   Test accuracy: {accuracy_score(y_test, clf_overfit.predict(X_test)):.3f}")
print("   (высокая точность на train, но низкая на test)")
overfit_tree = visualize_tree(
    clf_overfit,
    feature_names=['x1', 'x2'],
    class_names=['Class 0', 'Class 1'],
    title="Overfitted Decision Tree"
)

print("\n✅ ХОРОШЕЕ дерево (max_depth=3, min_samples_leaf=5):")
print(f"   Train accuracy: {accuracy_score(y_train, clf_good.predict(X_train)):.3f}")
print(f"   Test accuracy: {accuracy_score(y_test, clf_good.predict(X_test)):.3f}")
print("   (хороший баланс между train и test)")
good_tree = visualize_tree(
    clf_good,
    feature_names=['x1', 'x2'],
    class_names=['Class 0', 'Class 1'],
    title="Well-Generalized Decision Tree"
)

**Критерии остановы -  это предварительная стрижка (Pre-pruning)**

Кроме pre-pruning, существует **post-pruning:**

1. Сначала строим «полное» дерево (как будто не ограничиваем глубину)
2. Потом обрезаем некоторые ветви, которые не улучшают качество на валидации

### Cost-complexity pruning:

$$
R_\alpha(T) = R(T) + \alpha \cdot |T|
$$

**Где:**
- $R(T)$ — мера ошибки дерева (суммарная по листьям; можно думать как «empirical risk»)
- $|T|$ — размер дерева (например, число листьев)  
- $\alpha \geq 0$ — как сильно штрафуем сложность

**Логика:**
- При **большом $\alpha$** выгодно иметь маленькое дерево
- При **$\alpha \rightarrow 0$** сложность почти не штрафуется

**В sklearn:** этот подход реализован через параметр `ccp_alpha`.

In [ ]:
# Cost-Complexity Pruning (Post-pruning)

#Строим  дерево (без ограничений)
clf_full = DecisionTreeClassifier(random_state=0)
path = clf_full.cost_complexity_pruning_path(X_train, y_train)  # ← находим ВСЕ возможные α
ccp_alphas = path.ccp_alphas  # ← список штрафов за сложность (от 0 до большого)

print(f"Количество возможных деревьев: {len(ccp_alphas)}")
print(f"Первые 10 α: {ccp_alphas[:10]}")

# 2. Обучаем дерево для КАЖДОГО значения α (от полного до минимального)
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)  # ← α штрафует сложность
    clf.fit(X_train, y_train)
    clfs.append(clf)

# 3. Считаем метрики для всех деревьев
train_accs = [accuracy_score(y_train, clf.predict(X_train)) for clf in clfs]
test_accs = [accuracy_score(y_test, clf.predict(X_test)) for clf in clfs]
depths = [clf.get_depth() for clf in clfs]
leaves = [clf.get_n_leaves() for clf in clfs]

# 4. Таблица результатов
df_prune = pd.DataFrame({
    "ccp_alpha": ccp_alphas,
    "depth": depths,
    "leaves": leaves,
    "train_acc": train_accs,
    "test_acc": test_accs,
})
print(df_prune)

# 5. ГРАФИК: как α влияет на качество
plt.figure(figsize=(10, 6))
plt.semilogx(ccp_alphas, train_accs, "o-", label="Train accuracy")  # log-шкала для α
plt.semilogx(ccp_alphas, test_accs, "s-", label="Test accuracy")
plt.xlabel("ccp_alpha (штраф за сложность)")
plt.ylabel("Accuracy")
plt.title("Cost-Complexity Pruning: поиск оптимального α")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 🎯 ЛУЧШЕЕ α — где test_acc максимальна!
best_alpha = df_prune.loc[df_prune["test_acc"].idxmax(), "ccp_alpha"]
print(f"\nЛучшее α = {best_alpha:.6f} (test_acc = {df_prune['test_acc'].max():.3f})")



### 🧩 Задание 1. Эмпирические вероятности и энтропия узла

В этом задании вы закрепите понятие **эмпирической вероятности** и **энтропии**.

**Шаги:**  
1. Используйте массив меток `y_node`, заданный ниже.  
2. Найдите эмпирические вероятности классов.  
3. Вычислите энтропию узла.

Округлите результат до **3 знаков после запятой**.


In [ ]:
import numpy as np

# Массив меток в узле (0 и 1 — классы)
y_node = np.array([1, 1, 0, 1, 0, 0, 1, 1, 0, 1])

# Ваш код ниже
# Вычислите энтропию узла по формуле H = -∑ p_k * log2(p_k)
# =====================
p0 = np.mean(y_node == 0)
p1 = np.mean(y_node == 1)
H = -(p0 * np.log2(p0) + p1 * np.log2(p1))
# =====================

print("p0 =", round(p0, 3), ", p1 =", round(p1, 3))
print("H =", round(H, 3))


In [ ]:
# Проверка
assert np.isclose(p0 + p1, 1.0), "Сумма вероятностей должна быть равна 1"
assert round(H, 3) == 0.971, "Проверьте формулу энтропии — ожидается 0.971"
print("✅ Всё верно! Энтропия вычислена корректно.")


### 🌳 Задание 2. Сравнение критериев Джини и Энтропии

1. Обучите два дерева решений на одной и той же выборке:  
   - первое с `criterion='gini'`,  
   - второе с `criterion='entropy'`.

2. Используйте данные из `sklearn.datasets.make_classification`.

3. Сравните точность (`accuracy`) на тестовой выборке и сделайте вывод —  
   отличается ли результат в зависимости от критерия информативности?


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Генерация данных
X, y = make_classification(n_samples=500, n_features=4, n_informative=3, n_redundant=0, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Ваш код ниже
# =====================
tree_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_entropy = DecisionTreeClassifier(criterion='entropy', random_state=42)

tree_gini.fit(X_train, y_train)
tree_entropy.fit(X_train, y_train)
# =====================

acc_gini = accuracy_score(y_test, tree_gini.predict(X_test))
acc_entropy = accuracy_score(y_test, tree_entropy.predict(X_test))

print(f"Accuracy (Gini): {acc_gini:.3f}")
print(f"Accuracy (Entropy): {acc_entropy:.3f}")

**Вывод по заданию 2:** на этой выборке оба критерия дают близкое качество. Разница в accuracy может быть небольшой, потому что Gini и энтропия по-разному считают нечистоту, но обычно выбирают похожие разбиения. На практике критерий стоит сравнивать на валидации, особенно если качество моделей близко.

In [ ]:
# Проверка
assert 0.7 <= acc_gini <= 1.0, "Слишком низкая точность для Gini — проверьте обучение модели."
assert 0.7 <= acc_entropy <= 1.0, "Слишком низкая точность для Entropy — проверьте обучение модели."
print("✅ Модели обучены корректно! Можно сравнивать критерии.")

### ⚙️ Задание 3. Переобучение и глубина дерева
Проверьте, как параметр max_depth влияет на переобучение.

1. Обучите две модели:
    * tree_shallow с max_depth=2
    * tree_deep с max_depth=None (без ограничений)
2. Сравните точность на обучающей и тестовой выборке.
3. Сделайте вывод: какое дерево переобучилось?

In [ ]:
# Ваш код ниже
# =====================
tree_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_deep = DecisionTreeClassifier(max_depth=None, random_state=42)

tree_shallow.fit(X_train, y_train)
tree_deep.fit(X_train, y_train)
# =====================

train_acc_shallow = accuracy_score(y_train, tree_shallow.predict(X_train))
test_acc_shallow = accuracy_score(y_test, tree_shallow.predict(X_test))

train_acc_deep = accuracy_score(y_train, tree_deep.predict(X_train))
test_acc_deep = accuracy_score(y_test, tree_deep.predict(X_test))

print(f"Shallow tree: train={train_acc_shallow:.3f}, test={test_acc_shallow:.3f}")
print(f"Deep tree:    train={train_acc_deep:.3f}, test={test_acc_deep:.3f}")

**Вывод по заданию 3:** глубокое дерево без ограничения глубины лучше подстраивается под train и обычно показывает более высокий train accuracy. Если при этом test accuracy заметно ниже train accuracy, это признак переобучения. Ограничение `max_depth=2` делает дерево проще и снижает риск запоминания шума.

In [ ]:
# Проверка
assert train_acc_deep >= train_acc_shallow, "Ожидается, что глубокое дерево обучается лучше на train."
assert test_acc_deep <= train_acc_deep, "Глубокое дерево должно показывать признаки переобучения."
print("✅ Отлично! Эффект переобучения наблюдается.")